# Plot

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(rstatix)

options(tibble.width = Inf)

## Plot parameters

In [ ]:
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/blastoid_BF/output"
filter_dapi = TRUE

analysis_summary_files = c(
"/ceph.groups/mshahbazi.grp/rsakata/Figures/blastoid_BF/Evos_Count.csv")

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
color_condition = c("Control" = "#7BBA56", 
               "Reversine" = "#87549B", 
               "Mosaic" = "#E8973E")

col_Annexin= "#4db2cb"

col_ZVAD = c("ZVAD+" = "#6a6969", 
               "ZVAD-" = "#bbbbbb")

col_GFP = "#7BBA56"
col_RFP = "#cb377c"

col_structure= c("developed" = "#4674b8",  
        "small.cavity" ="#F09938" , 
        "failed" = "#fe941b")

## 1. Extract summary files

In [ ]:
merged_df <- read_csv(analysis_summary_files)


In [ ]:
head(merged_df)

In [ ]:
colnames(merged_df)

In [ ]:
unique(merged_df$condition)

In [ ]:
df_sample = merged_df

In [ ]:
melt_data =  melt(df_sample, id = c("condition"), measure.vars = c("developed",'small cavity','failed'),variable.name = "structure", value.name='frequency' )

melt_data$condition = factor(melt_data$condition, levels =unique(melt_data$condition))
head(melt_data)

# Plot 

### A) Proportion of structures 

In [ ]:
melt_data = melt_data %>%
filter(condition %in% c('control', 'mosaic', 'reversine'))%>%
filter(structure %in% c('developed', 'failed'))

In [ ]:

order_sample <- c(
'control', 'mosaic', 'reversine')

melt_data <- melt_data %>%
  mutate(condition = factor(condition, levels = order_sample))

In [ ]:
title = "proportion developed"
w <- 2.7
h <- 2
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(melt_data, aes(x = condition , y = frequency, fill  = structure)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75), alpha = 0.6, width = 0.6) +   # error bars
    geom_jitter(
      aes(fill = structure),
      position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
      size = 0.8, alpha = 0.9, color = "#505150"
    )  +  # average bar
    stat_summary(
      fun.data = mean_se, 
      geom = "errorbar",
      position = position_dodge(width = 0.75),
      width = 0.2, 
      color = "#505150")+
    labs(
      title = title,
      y = "proportion developed",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 1.10), expand = c(0, 0))+
      #facet_wrap( ~ condition_2) +
      scale_fill_manual(values=col_structure)

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p

In [ ]:
melt_data

In [ ]:
melt_data %>%
  group_by(condition) %>%
  t_test(
    frequency ~ structure,
    p.adjust.method = "none"
  ) 

## Contingency

In [ ]:
head(df_sample)

In [ ]:
df_sample_sub = df_sample %>%
filter(condition %in% c('control', 'mosaic', 'reversine'))

In [ ]:
contingency <- df_sample_sub %>%
  group_by(condition) %>%
  summarise(
    developed = sum(n_developed, na.rm = TRUE),
    failed = sum(n_failed, na.rm = TRUE),
    .groups = "drop"
  )

contingency

In [ ]:
# Create contingency table
tab <- matrix(
  c(
    239, 26,   # control
    204, 61,   # mosaic
    20,  205   # reversine
  ),
  nrow = 3,
  byrow = TRUE,
  dimnames = list(
    condition = c("control", "mosaic", "reversine"),
    outcome   = c("developed", "failed")
  )
)

tab

In [ ]:
fisher.test(tab)

In [ ]:
melt_data =  melt(contingency, id = c("condition"), measure.vars = c("developed",'failed'),variable.name = "structure", value.name='frequency' )

melt_data$condition = factor(melt_data$condition, levels =unique(melt_data$condition))
head(melt_data)

In [ ]:
df_percent <- melt_data  %>%
  group_by(condition) %>%
  mutate(
    total = sum(frequency),
    percentage = frequency / total * 100
  ) %>%
  ungroup()

df_percent

In [ ]:
title = "proportion developed_contingency"
w <- 2.7
h <- 1.7
options(repr.plot.width=w, repr.plot.height=h)
  
p = ggplot(df_percent, aes(x = condition , y = percentage, fill  = structure)) +  # dots for each file
    stat_summary(
      fun = mean, 
      geom = "bar",
      position = position_dodge(width = 0.75), alpha = 0.6, width = 0.6) +   # error bars
    labs(
      title = title,
      y = "proportion developed",
      x = "condition"
    )+ settheme+
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1),
      #legend.position = "none", 
      legend.key.size = unit(0.3, "cm")
      
    ) +
      scale_y_continuous(limits = c(0, 110), expand = c(0, 0))+
      #facet_wrap( ~ condition_2) +
      scale_fill_manual(values=col_structure)

ggsave(file.path(out_dir, sprintf("A_%s.pdf", title)),
                plot = p, width = w, height = h)

p